# TJAP - Tribunal de Justica do Amapa

Scraper for the jurisprudence search of the Court of Justice of the State of Amapa (TJAP).
Uses the [Tucujuris](https://tucujuris.tjap.jus.br) platform with a JSON REST API.

| Feature | Available |
|---------|-----------|
| cjsg    | Yes       |
| cpopg   | No        |
| cposg   | No        |

> **Blocked portal:** since 2026 the TJAP jurisprudence search requires a Cloudflare Turnstile CAPTCHA, validated by the server ([#279](https://github.com/jtrecenti/juscraper/issues/279)). The scraper cannot produce that token, so `cjsg` raises `TJAPSecurityCheckError`. From datacenter IPs, such as CI runners, the portal answers HTTP 403 before that and the call raises `RetryExhaustedError`. The examples below run only when the first search goes through.

In [ ]:
import juscraper as jus

# Create the TJAP scraper
tjap = jus.scraper('tjap')

## Basic search

In [ ]:
from juscraper.core.exceptions import RetryExhaustedError
from juscraper.courts.tjap.exceptions import TJAPSecurityCheckError

# Search for jurisprudence; dados_cjsg stays None when the portal blocks the search
dados_cjsg = None
try:
    dados_cjsg = tjap.cjsg('dano moral', paginas=range(1, 3))
except TJAPSecurityCheckError as erro:
    print(erro)
except RetryExhaustedError as erro:
    if erro.status_code != 403:
        raise
    print(f'Portal refused the request (HTTP 403): {erro}')

if dados_cjsg is not None:
    print(dados_cjsg.shape)
    display(dados_cjsg.head(3))

## Available columns

In [ ]:
if dados_cjsg is not None:
    print(dados_cjsg.columns.tolist())

## Preview ementa

In [ ]:
if dados_cjsg is not None:
    print(dados_cjsg['ementa'].iloc[0][:300])

## Using filters

The TJAP scraper supports several filters: orgao (court body), relator, classe, secretaria, origem, votacao, and more.

In [ ]:
# Filter by orgao (Tribunal only)
if dados_cjsg is not None:
    dados_filtrados = tjap.cjsg(
        'dano moral',
        orgao='tj',
        paginas=1,
    )
    print(dados_filtrados.shape)
    display(dados_filtrados.head(3))

## Download and parse separately

You can also download the raw JSON responses and parse them in a second step.

In [ ]:
# Download raw JSON responses
if dados_cjsg is not None:
    brutos = tjap.cjsg_download('dano moral', paginas=1)

    # Inspect the raw response structure
    print(f"Number of pages downloaded: {len(brutos)}")
    print(f"Status: {brutos[0]['status']}")
    print(f"Results in first page: {len(brutos[0]['dados'])}")

In [ ]:
# Parse raw responses into a DataFrame
if dados_cjsg is not None:
    df_parsed = tjap.cjsg_parse(brutos)
    print(df_parsed.shape)
    display(df_parsed.head(3))